In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                           f1_score, accuracy_score, precision_score, 
                           precision_recall_curve, recall_score, confusion_matrix)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

# Load the dataset
data = pd.read_excel(r'C:\Users\Inspiron\OneDrive - Loughborough University\Desktop\PhD\articles\prospective study\results\dataset\class 123\class123_dataset.xlsx')

# Prepare data with LASSO feature selection
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome
feature_indexes = [224, 3, 204, 138, 88, 21, 183, 22, 227, 142, 24, 244, 13, 103, 119, 184, 86, 26, 38, 155, 229, 222, 122, 253, 126, 174, 25, 82, 77, 137, 157, 182, 42, 216, 49, 232, 115, 133, 196, 231, 217, 90, 210, 116, 202, 59, 156, 109, 65, 20, 52, 201, 113, 68, 243, 214, 206, 185, 57, 197, 78, 64, 1, 221, 69, 80, 91, 11, 176, 29, 35, 123, 117, 33, 67, 147, 215, 61, 154, 207, 51, 252, 62, 190, 179, 30, 124, 167, 148, 28, 203, 168, 53, 146, 5]  # LASSO-selected features
X = X.iloc[:, feature_indexes]

# Define Logistic Regression with elasticnet penalty
classifier = LogisticRegression(
    C=1,                   # Reduced regularization strength from 30 to 1
    l1_ratio=0.3,          # Added elasticnet mixing parameter (30% L1, 70% L2)
    penalty='elasticnet',  # Changed from L2 to elasticnet
    solver='saga',         # Changed from sag to saga (supports elasticnet)
    random_state=42
)

# Define pipeline with updated RandomOverSampler
pipeline = Pipeline([
    ('resampler', RandomOverSampler(sampling_strategy=0.5, random_state=42)),  # Changed to explicit 0.5 ratio
    ('classifier', classifier)
])

# Initialize cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage variables
all_actuals = []
all_probs = []
auc_scores = []
auprc_scores = []

# Cross-validation loop
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probability of positive class
    
    # Store results for combined threshold calculation
    all_actuals.extend(y_test.values)
    all_probs.extend(y_prob)
    
    # Calculate fold metrics
    auc_scores.append(roc_auc_score(y_test, y_prob))
    auprc_scores.append(average_precision_score(y_test, y_prob))

# Find optimal threshold using all predictions
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

# Recalculate metrics with optimal threshold
f1_list, acc_list, prec_list, sens_list, spec_list = [], [], [], [], []
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    # Compute metrics
    f1_list.append(f1_score(y_test, y_pred))
    acc_list.append(accuracy_score(y_test, y_pred))
    prec_list.append(precision_score(y_test, y_pred, zero_division=0))
    sens_list.append(recall_score(y_test, y_pred))  # Sensitivity = Recall
    
    # Calculate specificity (TN / (TN + FP))
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp + 1e-9)  # Avoid division by zero
    spec_list.append(specificity)

# Calculate statistics
def format_metric(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("=== Logistic Regression with RandomOverSampler ===")
print(f"Optimal Threshold (F1-maximizing): {best_threshold:.4f}")
print(f"Average AUC: {format_metric(np.mean(auc_scores), np.std(auc_scores))}")
print(f"Average AUPRC: {format_metric(np.mean(auprc_scores), np.std(auprc_scores))}")
print(f"F1 Score: {format_metric(np.mean(f1_list), np.std(f1_list))}")
print(f"Accuracy: {format_metric(np.mean(acc_list), np.std(acc_list))}")
print(f"Precision: {format_metric(np.mean(prec_list), np.std(prec_list))}")
print(f"Sensitivity (Recall): {format_metric(np.mean(sens_list), np.std(sens_list))}")
print(f"Specificity: {format_metric(np.mean(spec_list), np.std(spec_list))}")

=== Logistic Regression with RandomOverSampler ===
Optimal Threshold (F1-maximizing): 0.5458
Average AUC: 0.7505 ± 0.0327
Average AUPRC: 0.2759 ± 0.0592
F1 Score: 0.3278 ± 0.0551
Accuracy: 0.8565 ± 0.0156
Precision: 0.2883 ± 0.0523
Sensitivity (Recall): 0.3826 ± 0.0650
Specificity: 0.9040 ± 0.0149
